# Revised experiments — Scientific Reports

This is the main revision notebook, building on the unchanged original `main.ipynb`.
The same Beijing PM2.5 task, three model families and two updating regimes remain the study’s basis. Supporting modules make corrections and results reproducible.

## Phase 1 — Data audit and correctness

Reports and artifacts are generated under `revision/`. Bounded pilots are feasibility checks, not final comparisons.


In [1]:
from pathlib import Path
import subprocess, sys, os
root = Path.cwd()
if not (root / "main.ipynb").exists():
    root = next(p for p in root.parents if (p / "main.ipynb").exists())
python = str(root / "venv/bin/python") if (root / "venv/bin/python").exists() else sys.executable
env = os.environ.copy()
for name in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    env[name] = "2"
env["MPLCONFIGDIR"] = "/tmp/weather-revision-matplotlib"
def run(*args):
    return subprocess.run([python, *args], cwd=root, env=env, check=True)


In [2]:
run("-m", "revision.code.audit")


Out[2]: CompletedProcess(args=['/home/moazzam/Documents/Windows/VS Code/Python/Research/Weather Forecasting - Experiments/venv/bin/python', '-m', 'revision.code.audit'], returncode=0)
{
  "raw_rows": 39523,
  "raw_missing_values": 0,
  "calendar_hours": 40267,
  "missing_hours": 744,
  "gap_intervals": 21,
  "training_calendar_hours": 36240,
  "training_observed_targets": 35736,
  "test_calendar_hours": 4027,
  "test_observed_targets": 3787,
  "full_test_windows": 23,
  "eligible_test_windows": 16,
  "evaluated_hours": 2688,
  "tail_hours": 163,
  "preprocessed_rows": 36830,
  "removed_original_rows": 2693,
  "preprocessed_matches_notebook_global_zscore_filter": true
}
Eligible test windows: [1, 2, 3, 4, 5, 6, 7, 10, 16, 17, 18, 19, 20, 21, 22, 23]


 *** This program and the respective minimum Redundancy Maximum Relevance (mRMR) 
     algorithm were developed by Hanchuan Peng <hanchuan.peng@gmail.com>for
     the paper 
     "Feature selection based on mutual information: criteria of 

In [3]:
run("-m", "unittest", "discover", "-s", "revision/tests", "-v")


Out[3]: CompletedProcess(args=['/home/moazzam/Documents/Windows/VS Code/Python/Research/Weather Forecasting - Experiments/venv/bin/python', '-m', 'unittest', 'discover', '-s', 'revision/tests', '-v'], returncode=0)
test_neural_origin_uses_scaled_history_and_future_without_future_y (test_phase2.Phase2Tests.test_neural_origin_uses_scaled_history_and_future_without_future_y) ... ok
test_saved_stream_rejects_wrong_week_and_false_observation_mask (test_phase2.Phase2Tests.test_saved_stream_rejects_wrong_week_and_false_observation_mask) ... ok
test_validation_grid_has_all_bounded_candidates (test_phase2.Phase2Tests.test_validation_grid_has_all_bounded_candidates) ... ok
test_weekly_rows_rejects_missing_score_target (test_phase2.Phase2Tests.test_weekly_rows_rejects_missing_score_target) ... ok
test_bias_is_applied_before_current_outcomes_update_it (test_protocol.ProtocolTests.test_bias_is_applied_before_current_outcomes_update_it) ... ok
test_duplicate_origins_and_nonfinite_predictions_are_err

## Installed-library checks and feasibility pilots

Run only after reviewing the data audit and protocol. These may take several minutes. Pilot predictions are training-only validation forecasts and must not be presented as final model comparisons.


In [4]:
# run("-m", "revision.code.model_checks")
# run("-m", "revision.code.pilots")


In [5]:
import pandas as pd
windows = pd.read_csv(root / "revision/artifacts/phase1/test_windows.csv")
windows


Out[5]: 
    window  ...                                          exclusion
0        1  ...                                                NaN
1        2  ...                                                NaN
2        3  ...                                                NaN
3        4  ...                                                NaN
4        5  ...                                                NaN
5        6  ...                                                NaN
6        7  ...                                                NaN
7        8  ...             missing_future_target_or_PP_covariates
8        9  ...                                    missing_context
9       10  ...                                                NaN
10      11  ...             missing_future_target_or_PP_covariates
11      12  ...  missing_context;missing_future_target_or_PP_co...
12      13  ...                                    missing_context
13      14  ...             missing_future_target_or_

## Phase 2 — Training-only validation and matched comparison

The experiments run through `revision.code.phase2_runner` in isolated, sequential processes. Moazzam does not need to execute this notebook. The runner checkpoints each valid forecast origin; the saved CSV and JSON files below are the evidence used for the paper and response. `main.ipynb` remains untouched.

First, four complete validation weeks inside the training period screen four SARIMAX orders, two Prophet seasonality modes, and two NeuralProphet epoch counts. Only the selected settings go into the held-out comparison. The held-out evaluation uses 16 complete, matched weeks (2,688 hours). Refit and frozen runs, persistence baselines, seed repeats, predictor ablations, clipping sensitivity, and EWMA alternatives share the established calendar and information set.

The following commands are managed and run by the revision workflow; this notebook reads their artifacts after completion:

```bash
venv/bin/python -m revision.code.phase2_runner validate
venv/bin/python -m revision.code.phase2_runner baselines
venv/bin/python -m revision.code.phase2_runner core
venv/bin/python -m revision.code.phase2_runner ablate
venv/bin/python -m revision.code.phase2_runner correct
venv/bin/python -m revision.code.phase2_recover
venv/bin/python -m revision.code.phase2_summary
venv/bin/python -m revision.code.phase2_status
```

Validation must complete before `core` or `ablate`. Repeating a command skips completed tasks whose protocol/code signature matches. Check `artifacts/phase2/supervisor.jsonl` and each task's `metadata.json` and `run.log` for failures before interpreting a table.

Training-only validation is complete: SARIMAX `(1,0,1) × (1,0,1,24)`, additive Prophet and 50-epoch NeuralProphet were selected. Three of four SARIMAX candidates converged; the failed candidate is preserved in its metadata. The scheduled held-out matrix has finished. Separately recorded numerical recoveries address nonconverged SARIMAX fits without changing selected model settings or information sets. The verification cell below reports current accepted coverage. Original failed attempts remain in the audit trail.


In [6]:
import json
phase2 = root / "revision/artifacts/phase2"
selection_file = phase2 / "selection.json"
if selection_file.exists():
    selected = json.loads(selection_file.read_text())
    display(pd.DataFrame([{"family": family, **candidate} for family, candidate in selected["selected"].items()]))
else:
    print("Training-only validation is still running; no model settings are selected yet.")


,family,order,seasonal_order,seasonality_mode,epochs
0,sarimax,"[1, 0, 1]","[1, 0, 1, 24]",NaN,NaN
1,prophet,NaN,NaN,additive,NaN
2,neuralprophet,NaN,NaN,NaN,50.0


In [7]:
from pathlib import Path
runs = phase2 / "runs"
records = []
for metadata_file in sorted(runs.glob("*/metadata.json")) if runs.exists() else []:
    record = json.loads(metadata_file.read_text())
    task = record["task"]
    records.append({"run_id":task["id"],"stage":task["stage"],"family":task["family"],
                    "status":record.get("status"),"scored_hours":record.get("scored_hours"),
                    "mean_weekly_mae":record.get("mean_weekly_mae"),
                    "pooled_rmse":record.get("pooled_rmse")})
display(pd.DataFrame(records))


,run_id,stage,family,status,scored_hours,mean_weekly_mae,pooled_rmse
0,ablate_neuralprophet_broad,ablate,neuralprophet,completed,2688.0,20.868162,28.780764
1,ablate_neuralprophet_clip,ablate,neuralprophet,completed,2688.0,60.012240,78.662211
2,ablate_neuralprophet_no_inputs,ablate,neuralprophet,completed,2688.0,127.462355,192.971240
3,ablate_prophet_broad,ablate,prophet,completed,2688.0,19.923080,26.648186
4,ablate_prophet_clip,ablate,prophet,completed,2688.0,48.279714,64.632300
...,...,...,...,...,...,...,...
103,validate_prophet_1,validate,prophet,completed,672.0,41.500746,56.988164
104,validate_sarimax_0,validate,sarimax,completed,672.0,15.024757,22.369720
105,validate_sarimax_1,validate,sarimax,failed,NaN,NaN,NaN
106,validate_sarimax_2,validate,sarimax,completed,672.0,38.941065,58.326836


## Verified evidence and descriptive results

The coverage mapping names the exact original or recovered forecast used for each planned task. Convergence, calendar/lead alignment, observed targets, correction timing, and original-file checksums are checked by `revision.code.phase2_verify`. Numerical recoveries retain their own provenance and resource records.

These are descriptive point estimates. Phase 3 will provide paired uncertainty, lead-time diagnostics, residual analysis, seed variability, and publication figures. The manuscript and point-by-point response are still pending.


In [8]:
verification = json.loads((phase2 / "verification.json").read_text())
print(json.dumps({key: verification[key] for key in [
    "complete_tasks", "expected_core_ablation_tasks", "recovered_tasks",
    "unresolved_failed_tasks", "integrity_errors", "validation_completed",
    "validation_failures", "baselines", "correction_streams", "original_files_verified"
]}, indent=2))
coverage = pd.read_csv(phase2 / "verified_coverage.csv")
display(coverage.groupby(["family", "regime", "variant", "seed"])[["windows", "hours", "recovered_windows"]].sum())


{
  "complete_tasks": 94,
  "expected_core_ablation_tasks": 94,
  "recovered_tasks": 6,
  "unresolved_failed_tasks": 0,
  "integrity_errors": [],
  "validation_completed": 7,
  "validation_failures": [
    "validate_sarimax_1"
  ],
  "baselines": 3,
  "correction_streams": 35,
  "original_files_verified": 21
}


windows  hours  recovered_windows
family        regime variant   seed                                   
neuralprophet frozen broad     42         16   2688                  0
                     clip      42         16   2688                  0
                     no_inputs 42         16   2688                  0
                     selected  42         16   2688                  0
                               123        16   2688                  0
                               2026       16   2688                  0
              walk   selected  42         16   2688                  0
                               123        16   2688                  0
                               2026       16   2688                  0
prophet       frozen broad     42         16   2688                  0
                     clip      42         16   2688                  0
                     no_inputs 42         16   2688                  0
                     selected  42         16   2688                  0
              walk   selected  42         16   2688                  0
sarimax       frozen broad     42         16   2688                  0
                     clip      42         16   2688                 16
                     no_inputs 42         16   2688                  0
                     selected  42         16   2688                  0
              walk   selected  42         16   2688                  5

In [9]:
metrics = pd.read_csv(phase2 / "experimental_summary.csv")
display(metrics[metrics.variant.eq("selected")])


,family,regime,variant,seed,recovered_windows,windows,hours,pooled_mae,pooled_rmse,mean_weekly_mae,weekly_mae_sd,mean_weekly_rmse,weekly_rmse_sd
3,neuralprophet,frozen,selected,42,0,16,2688,38.442519,54.157017,38.442519,18.560054,48.934618,23.964100
4,neuralprophet,frozen,selected,123,0,16,2688,40.701115,55.787384,40.701115,17.572055,51.358198,22.499014
5,neuralprophet,frozen,selected,2026,0,16,2688,34.456326,47.357530,34.456326,13.556378,43.778159,18.653569
6,neuralprophet,walk,selected,42,0,16,2688,34.224598,48.929429,34.224598,14.612986,44.298301,21.460030
7,neuralprophet,walk,selected,123,0,16,2688,36.039326,50.488805,36.039326,13.966784,46.460927,20.409193
8,neuralprophet,walk,selected,2026,0,16,2688,34.250126,47.300391,34.250126,13.693926,43.613797,18.907880
12,prophet,frozen,selected,42,0,16,2688,36.963392,49.281056,36.963392,12.948512,46.531975,16.761791
13,prophet,walk,selected,42,0,16,2688,35.497585,48.594115,35.497585,12.453394,45.621826,17.283117
17,sarimax,frozen,selected,42,0,16,2688,29.914284,47.541491,29.914284,17.300945,40.733084,25.319488
18,sarimax,walk,selected,42,5,16,2688,29.877627,47.505589,29.877627,17.295168,40.679786,25.339002


In [10]:
display(metrics[metrics.regime.eq("frozen") & metrics.seed.eq(42)])


,family,regime,variant,seed,recovered_windows,windows,hours,pooled_mae,pooled_rmse,mean_weekly_mae,weekly_mae_sd,mean_weekly_rmse,weekly_rmse_sd
0,neuralprophet,frozen,broad,42,0,16,2688,20.868162,28.780764,20.868162,5.879963,27.177300,9.783073
1,neuralprophet,frozen,clip,42,0,16,2688,60.012240,78.662211,60.012240,32.119543,72.091776,32.504675
2,neuralprophet,frozen,no_inputs,42,0,16,2688,127.462355,192.971240,127.462355,102.995697,152.731490,121.812934
3,neuralprophet,frozen,selected,42,0,16,2688,38.442519,54.157017,38.442519,18.560054,48.934618,23.964100
9,prophet,frozen,broad,42,0,16,2688,19.923080,26.648186,19.923080,6.171178,25.488139,8.031975
10,prophet,frozen,clip,42,0,16,2688,48.279714,64.632300,48.279714,17.691605,61.242370,21.334325
11,prophet,frozen,no_inputs,42,0,16,2688,146.841107,208.317071,146.841107,99.162261,174.757483,117.102002
12,prophet,frozen,selected,42,0,16,2688,36.963392,49.281056,36.963392,12.948512,46.531975,16.761791
14,sarimax,frozen,broad,42,0,16,2688,17.572708,24.426489,17.572708,5.517022,22.896743,8.787418
15,sarimax,frozen,clip,42,16,16,2688,32.654833,52.168174,32.654833,16.128050,46.123453,25.174494


In [11]:
display(pd.read_csv(phase2 / "baseline_summary.csv"))
correction = pd.read_csv(phase2 / "corrections/summary.csv")
display(correction[correction.alpha.isin([0., 0.3, 1.])])


,run,windows,hours,mean_weekly_mae,weekly_mae_sd,pooled_rmse
0,daily_persistence,16,2688,131.184461,117.025894,222.425082
1,persistence,16,2688,125.372671,108.134489,203.799541
2,weekly_persistence,16,2688,164.586462,132.046347,268.420651


,run_id,alpha,mean_weekly_mae,pooled_rmse
0,core_sarimax_frozen_s42,0.0,29.914284,47.541491
3,core_sarimax_frozen_s42,0.3,29.478728,45.274477
6,core_sarimax_frozen_s42,1.0,33.077341,48.675804
7,core_prophet_frozen_s42,0.0,36.963392,49.281056
10,core_prophet_frozen_s42,0.3,35.743806,48.771906
13,core_prophet_frozen_s42,1.0,36.809893,49.235660
14,core_neuralprophet_frozen_s42,0.0,38.442519,54.157017
17,core_neuralprophet_frozen_s42,0.3,33.302983,48.054795
20,core_neuralprophet_frozen_s42,1.0,33.167404,47.527849
21,core_neuralprophet_frozen_s123,0.0,40.701115,55.787384
